# Lilly — train the Bosnian to English model

Before running anything, set three things in the panel on the right:

- **Session options → Accelerator → GPU T4 x2** (or P100)
- **Session options → Internet → On** — the notebook clones and downloads
- **Input → Add Input → Datasets** → the copy of `models/lilly/translate` you
  uploaded. Upload it once: Kaggle → Datasets → New Dataset → drag the folder in.
  The weights are too big for git, so this is how they reach the machine.

Then **Save Version → Save & Run All (Commit)** and close the tab. It keeps
running without you — about 2-3 hours. The finished adapter waits in the
Output tab of that version.


In [ ]:
# 1. Check we actually got a GPU
!nvidia-smi -L

In [ ]:
# 2. Get the Lilly code
%cd /kaggle/working
!rm -rf Lilly && git clone -q https://github.com/ssaaffaakk/Lilly.git
%cd /kaggle/working/Lilly

In [ ]:
# 3. Install what we need (~2 min) — same versions as on the Mac
!pip -q install transformers==4.49.0 peft==0.14.0 accelerate==1.3.0 \
    sacrebleu sentencepiece sacremoses

In [ ]:
# 4. Find the base weights among the datasets you attached
import glob, os
found = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/source.spm', recursive=True)]
if not found:
    raise SystemExit('Attach your models/lilly/translate folder as a Dataset input first '
                     '(right panel -> Input -> Add Input -> Datasets)')
os.environ['LILLY_BASE'] = found[0]
print('base weights:', found[0])

In [ ]:
# 5. Download and clean the Bosnian-English data (~5 min)
!python3 data/scripts/download_data.py
!python3 data/scripts/clean_data.py

In [ ]:
# 6. Quick pipeline check (~3 min) — tiny run, just to prove everything works
!python3 training/train_translation.py --quick-test

In [ ]:
# 7. THE REAL TRAINING (~2-3 hours)
!python3 training/train_translation.py

In [ ]:
# 8. Score it: untuned base vs our Lilly, on sentences it never saw (~20 min)
!python3 training/evaluate.py --adapter models/lilly/adapter --limit 500
!cat training/RESULTS.md

In [ ]:
# 9. Package the result so it survives the run (~20-40 MB)
!cd /kaggle/working/Lilly && zip -qr /kaggle/working/lilly-adapter.zip \
    models/lilly/adapter training/RESULTS.md
!ls -lh /kaggle/working/lilly-adapter.zip

**Done.** Open the finished version's **Output** tab and download
`lilly-adapter.zip`. Unzip it so the adapter sits at `models/lilly/adapter/` and
the app uses it the next time it starts.

Check `RESULTS.md` before you trust it: if the tuned numbers are not above the
base numbers, the run did not help and there is no point shipping it.
